# Predicting Rejected OMR Answer Sheets — Colab Notebook

Complete reproducible workflow for the OMR scanning rejection prediction MLOps project.

In [ ]:
# Install dependencies
!pip -q install pandas numpy scikit-learn joblib matplotlib seaborn fastapi uvicorn pydantic pytest httpx mlflow reportlab

In [ ]:
# Colab bootstrap: the notebook needs the repository's src/ package.
# If you uploaded only this .ipynb file, upload the complete project ZIP when prompted.
from pathlib import Path
import sys, zipfile, os

ROOT = Path('/content/omr-mlops-project')
if not (ROOT / 'src').exists():
    try:
        from google.colab import files
        print('Upload omr-mlops-project.zip downloaded from the project deliverables.')
        uploaded = files.upload()
        zip_names = [name for name in uploaded if name.lower().endswith('.zip')]
        if not zip_names:
            raise FileNotFoundError('Please upload the complete omr-mlops-project.zip file, not only the notebook.')
        with zipfile.ZipFile(zip_names[0]) as z:
            z.extractall('/content')
    except ImportError:
        print('Not running in Colab. Run this notebook from the project root.')

if not (ROOT / 'src').exists():
    raise FileNotFoundError('Project package not found. Upload and extract omr-mlops-project.zip, then rerun this cell.')
sys.path.insert(0, str(ROOT))
os.chdir(ROOT)
print('Using project root:', ROOT)

## 1. Dataset creation and version management

In [ ]:
from src.generate_data import generate
v1,v2=generate()
print('V1:',v1.shape,'V2:',v2.shape)
print('V1 rejection rate:',(v1.scan_rejected=='Yes').mean())
print('V2 rejection rate:',(v2.scan_rejected=='Yes').mean())

In [ ]:
v1[['dpi','rotation_angle','print_contrast_score']].describe()

## 2. Automated preprocessing and reusable feature store

In [ ]:
from src.features import create_feature_store, ROOT
features=create_feature_store(ROOT/'data/omr_scans_all.csv')
features[['scan_quality_index','marking_error_ratio','scan_rejected']].head()

In [ ]:
# Reuse the persisted feature file in a downstream step
import pandas as pd
reused=pd.read_csv(ROOT/'artifacts/feature_store.csv')
print('Reused feature store shape:',reused.shape)

## 3. Model training, comparison and MLflow registration

In [ ]:
from src.train import train_models
metrics=train_models()
metrics

In [ ]:
import matplotlib.pyplot as plt
metrics.set_index('model')[['accuracy','precision','recall','f1']].plot(kind='bar',ylim=(0,1),figsize=(9,5),title='Model comparison')
plt.xticks(rotation=0); plt.tight_layout()

## 4. FastAPI prediction

In [ ]:
from fastapi.testclient import TestClient
from src.api import app
client=TestClient(app)
request={'dpi':300,'rotation_angle':4.2,'print_contrast_score':52,'alignment_marker_visibility':.55,'multiple_marked_questions':4,'blank_answer_percentage':18,'sheet_damage_score':.35,'scanner_type':'legacy'}
response=client.post('/predict',json=request)
print(response.status_code,response.json())

## 5. CI/CD validation

Run `pytest -q`; GitHub Actions executes the same pipeline and tests on every push and pull request.